# AgentDecompile Usage Validation

 > Local-first, credential-free validation notebook for the executable parts of `USAGE.md`.

## Safety Contract

- Everything is read-only by default.
- Shared-server workflows are documentation-only unless explicitly enabled.
- Version-control-sensitive tools such as `checkout-program` and `checkin-program` are probed only to confirm they fail clearly in unsupported local contexts.

## Coverage

- Kernel and package validation
- Usage-doc command inventory
- Local fixture generation using repository helpers
- Local MCP server startup and discovery
- Local binary import/open and inspection flows
- Export and raw tool calls where supported
- Explicit version-control behavior checks

## Notebook Strategy

This notebook translates the docs into a runnable local harness wherever possible.

- `USAGE.md` is treated as the source of truth for command intent.
- Shared-server examples stay visible for onboarding, but they are not executed in the default run.
- Results are collected into a single summary table so mismatches between documented and observed behavior are obvious.

The notebook prefers native Jupyter features where they help:

- top-level `await` for MCP calls
- rich markdown and HTML summaries
- widgets for execution flags
- explicit cleanup cells so reruns stay predictable

## Live Contract Notes

This notebook now sits alongside a stricter terminal-validated E2E contract suite for the local server path.

- The default live local `tools/list` surface is currently 36 tools.
- Hidden compatibility tools such as `manage-comments` are still callable even though they are not advertised by default.
- `switch-project` still routes to `open-project` as a compatibility alias, but it is intentionally hidden from the default advertised surface.
- The strict local contract suite asserts the observed JSON payloads for `open-project`, `import-binary`, `list-project-files`, `get-current-program`, `list-functions`, `search-symbols`, `get-references`, `list-imports`, `list-exports`, `checkout-status`, `sync-project`, comment round-trips, and export artifacts.
- The current observed local `change-processor` behavior on `tests/fixtures/test_x86_64` is a failure response that leaves the active program unchanged; the suite records that exact contract so regressions are visible.

See `tests/test_e2e_local_terminal_contracts.py` and `examples/mcp_responses/local_live_contract_test_x86_64.json` for the captured reference used by those assertions.

## Remote Shared-Server Notes

This notebook stays local-first, but the docs it validates now include a live remote shared-server workflow that was re-verified during this session.

- Prefer `http://host:port/mcp` for MCP clients, CLI examples, and copied commands.
- `http://host:port/mcp/message` remains a compatibility path.
- `http://host:port/` and `http://host:port/api` are metadata/index routes, not alternate MCP transport routes.
- `/api/mcp` is not part of the supported endpoint surface.

The shared workflow that was re-verified against a live remote deployment covered these commands and outcomes:

- `list-project-files` returned the shared repository entries `/K1` and `/K1/k1_win_gog_swkotor.exe`.
- `get-current-program --program_path /K1/k1_win_gog_swkotor.exe` returned `swkotor.exe`, `x86:LE:32:default`, `windows`, and `24591` functions.
- `search-symbols --program_path /K1/k1_win_gog_swkotor.exe --query main` returned 58 matches including `WinMain`.
- `get-functions --program_path /K1/k1_win_gog_swkotor.exe --identifier WinMain` resolved `WinMain` at `004041f0`.

Fresh CLI invocations still create fresh MCP sessions, so the reopened-program behavior documented in `USAGE.md` matters for shared workflows.

## Session-Validated Commands

This notebook now records the exact AgentDecompile command shapes exercised during the documentation session that refreshed the surrounding markdown docs.

```powershell
# Published Docker image in stdio mode
docker run --rm -i \
  --add-host host.docker.internal:host-gateway \
  --entrypoint /ghidra/venv/bin/agentdecompile-server \
  docker.io/bolabaden/agentdecompile-mcp:latest \
  -t stdio

# Local-checkout CLI validation used in this session
$env:PYTHONPATH='src'
C:/GitHub/agentdecompile/.venv/Scripts/python.exe -m agentdecompile_cli.cli --server-url http://127.0.0.1:8097 tool-seq '[{"name":"open-project","arguments":{"path":"LocalRepo","serverHost":"127.0.0.1","serverPort":13100,"serverUsername":"<redacted>","serverPassword":"<redacted>","format":"json"}},{"name":"list-project-files","arguments":{"format":"json"}},{"name":"import-binary","arguments":{"path":"C:/GitHub/agentdecompile/tests/fixtures/test_x86_64","enableVersionControl":true,"format":"json"}},{"name":"list-project-files","arguments":{"format":"json"}},{"name":"remove-program-binary","arguments":{"programPath":"test_x86_64","confirm":true,"format":"json"}},{"name":"list-project-files","arguments":{"format":"json"}}]'
```

Equivalent user-facing form for the second command:

```powershell
uv run agentdecompile-cli --server-url http://127.0.0.1:8097 tool-seq '[{"name":"open-project","arguments":{"path":"LocalRepo","serverHost":"127.0.0.1","serverPort":13100,"serverUsername":"<redacted>","serverPassword":"<redacted>","format":"json"}},{"name":"list-project-files","arguments":{"format":"json"}},{"name":"import-binary","arguments":{"path":"C:/GitHub/agentdecompile/tests/fixtures/test_x86_64","enableVersionControl":true,"format":"json"}},{"name":"list-project-files","arguments":{"format":"json"}},{"name":"remove-program-binary","arguments":{"programPath":"test_x86_64","confirm":true,"format":"json"}},{"name":"list-project-files","arguments":{"format":"json"}}]'
```

The `tool-seq` run above exercised `open-project`, `list-project-files`, `import-binary`, and `remove-program-binary` in a single MCP session.

In [1]:
import asyncio
import html
import json
import os
import re
import shutil
import socket
import subprocess
import sys
import tempfile
import time
from pathlib import Path
from typing import Any

import nest_asyncio
from IPython.display import HTML, JSON, Markdown, display

nest_asyncio.apply()

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "USAGE.md").exists():
            return candidate
    raise RuntimeError(f"Could not locate repository root from {start}")

REPO_ROOT = find_repo_root(Path.cwd())
NOTEBOOK_PATH = REPO_ROOT / "examples" / "usage_validation.ipynb"
USAGE_PATH = REPO_ROOT / "USAGE.md"
TMP_ROOT = REPO_ROOT / "tmp" / "usage_validation"
TMP_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS = []

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
src_path = REPO_ROOT / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

def as_json(data):
    display(JSON(data, expanded=False))

def show_note(title: str, body: str, color: str = "#2855aa") -> None:
    display(
        HTML(
            f"<div style='border-left:4px solid {color}; padding:0.75rem 1rem; margin:0.5rem 0; background:#f7f9fc;'>"
            f"<strong>{html.escape(title)}</strong><br>{body}</div>"
        )
    )

usage_text = USAGE_PATH.read_text(encoding="utf-8")
env_report = {
    "repo_root": str(REPO_ROOT),
    "notebook": str(NOTEBOOK_PATH),
    "python_executable": sys.executable,
    "python_version": sys.version.split()[0],
    "usage_md_exists": USAGE_PATH.exists(),
    "shared_repo_quick_sequence_present": "### Shared repository quick sequence with uvx" in usage_text,
    "ghidra_install_dir": os.environ.get("GHIDRA_INSTALL_DIR"),
    "project_path_env": os.environ.get("AGENT_DECOMPILE_PROJECT_PATH"),
}

show_note(
    "Environment ready",
    "Kernel helpers, top-level await support, repo root detection, and import paths are initialized.",
    color="#1f6f43",
)
as_json(env_report)

<IPython.core.display.JSON object>

In [12]:
import ipywidgets as widgets

RUN_SHARED_SERVER_DOCS = widgets.Checkbox(value=False, description="Show shared-server notes")
RUN_VERSION_CONTROL_PROBES = widgets.Checkbox(value=True, description="Probe local checkout/checkin behavior")
VERBOSE_TOOL_OUTPUT = widgets.Checkbox(value=False, description="Verbose captured output")

def current_flags() -> dict[str, bool]:
    return {
        "show_shared_server_notes": bool(RUN_SHARED_SERVER_DOCS.value),
        "probe_version_control_behavior": bool(RUN_VERSION_CONTROL_PROBES.value),
        "verbose_tool_output": bool(VERBOSE_TOOL_OUTPUT.value),
    }

controls = widgets.VBox(
    [
        widgets.HTML("<h4 style='margin:0'>Execution Flags</h4><p style='margin:0'>These affect notebook display and the explicit local behavior probes.</p>"),
        RUN_SHARED_SERVER_DOCS,
        RUN_VERSION_CONTROL_PROBES,
        VERBOSE_TOOL_OUTPUT,
    ]
 )

display(controls)
as_json(current_flags())

<IPython.core.display.JSON object>

In [2]:
def extract_command_lines(path: Path) -> list[str]:
    lines = path.read_text(encoding="utf-8").splitlines()
    interesting = []
    for line in lines:
        stripped = line.strip()
        if any(token in stripped for token in ["agentdecompile-cli", "agentdecompile-server", "agentdecompile-proxy", "uvx --from", "call_tool", "Invoke-McpTool"]):
            interesting.append(stripped)
    return interesting

usage_commands = extract_command_lines(USAGE_PATH)
usage_text = USAGE_PATH.read_text(encoding="utf-8")

command_summary = {
    "USAGE.md command lines": len(usage_commands),
    "shared repository quick sequence present": "### Shared repository quick sequence with uvx" in usage_text,
}

sample_commands = usage_commands[:12]
rows = "".join(
    f"<tr><td style='padding:0.35rem 0.5rem; vertical-align:top'>{index}</td><td style='padding:0.35rem 0.5rem'><code>{html.escape(command)}</code></td></tr>"
    for index, command in enumerate(sample_commands, start=1)
 )

display(Markdown("## Doc Command Inventory"))
as_json(command_summary)
display(
    HTML(
        "<table style='border-collapse:collapse; width:100%'>"
        "<thead><tr><th style='text-align:left; padding:0.35rem 0.5rem'>#</th><th style='text-align:left; padding:0.35rem 0.5rem'>Sample command</th></tr></thead>"
        f"<tbody>{rows}</tbody></table>"
    )
)

## Doc Command Inventory

<IPython.core.display.JSON object>

#,Sample command
1,uvx --from /path/to/agentdecompile/ --with-editable /path/to/agentdecompile/ agentdecompile-cli ...
2,uvx --from /path/to/agentdecompile/ --with-editable /path/to/agentdecompile/ agentdecompile-server ...
3,uvx --from /path/to/agentdecompile/ --with-editable /path/to/agentdecompile/ agentdecompile-proxy ...
4,"- Add `--verbose` to `agentdecompile-cli`, `agentdecompile-server`, `agentdecompile-proxy`, or `mcp-agentdecompile` when you need transport diagnostics."
5,uvx --from git+https://github.com/bolabaden/agentdecompile mcp-agentdecompile
6,uvx --from git+https://github.com/bolabaden/agentdecompile agentdecompile-server -t streamable-http --project-path ./agentdecompile_projects
7,### Proxy mode (forward to remote MCP; use agentdecompile-proxy only)
8,uvx --from git+https://github.com/bolabaden/agentdecompile agentdecompile-proxy --backend-url http://***:8080 -t streamable-http --host 127.0.0.1 --port 8081
9,Or set `AGENT_DECOMPILE_MCP_SERVER_URL` or `AGENTDECOMPILE_MCP_SERVER_URL` and run `agentdecompile-proxy -t streamable-http`. **agentdecompile-server** is always local (PyGhidra) and does not accept proxy options.
10,"- If you are testing workspace edits, do not use `uvx --from git+https://github.com/bolabaden/agentdecompile ...` because that launches the packaged GitHub build, not your local checkout."


In [2]:
from tests.helpers import create_minimal_binary

FIXTURE_DIR = REPO_ROOT / "tests" / "fixtures"
SESSION_DIR = Path(tempfile.mkdtemp(prefix="usage_validation_", dir=str(TMP_ROOT)))
PROJECT_DIR = SESSION_DIR / "project"
OUTPUT_DIR = SESSION_DIR / "exports"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_BINARY = SESSION_DIR / "minimal_test_elf.bin"
create_minimal_binary(LOCAL_BINARY)

fixture_candidates = [
    FIXTURE_DIR / "test_x86_64",
    FIXTURE_DIR / "test_arm64",
    FIXTURE_DIR / "test_fat_binary",
    LOCAL_BINARY,
 ]
available_binaries = [path for path in fixture_candidates if path.exists()]
PRIMARY_BINARY = available_binaries[0]

SESSION_STATE = {
    "session_dir": str(SESSION_DIR),
    "project_dir": str(PROJECT_DIR),
    "output_dir": str(OUTPUT_DIR),
    "primary_binary": str(PRIMARY_BINARY),
    "available_binaries": [str(path) for path in available_binaries],
}

show_note(
    "Fixture workspace ready",
    f"Primary local binary: <code>{html.escape(str(PRIMARY_BINARY))}</code>",
    color="#7a4a00",
)
as_json(SESSION_STATE)

<IPython.core.display.JSON object>

In [ ]:
from agentdecompile_cli.bridge import AgentDecompileMcpClient

SERVER_PROCESS = None
SERVER_LOG_HANDLE = None
SERVER_INFO = {}
CURRENT_PROGRAM_PATH = None
TOOL_NAMES = []

TOOL_ALIASES = {
    "open": ["open-project", "open_project", "import-binary", "import_binary"],
    "open-project": ["open-project", "open_project", "import-binary", "import_binary"],
    "import-binary": ["import-binary", "import_binary"],
    "list-project-files": ["list-project-files", "list_project_files"],
    "get-current-program": ["get-current-program", "get_current_program"],
    "list-functions": ["list-functions", "list_functions"],
    "get-functions": ["get-functions", "get_functions"],
    "search-symbols-by-name": ["search-symbols-by-name", "search_symbols_by_name"],
    "get-references": ["get-references", "get_references", "references"],
    "export": ["export"],
    "list-imports": ["list-imports", "list_imports"],
    "list-exports": ["list-exports", "list_exports"],
    "manage-strings": ["manage-strings", "manage_strings"],
    "search-constants": ["search-constants", "search_constants"],
    "get-call-graph": ["get-call-graph", "get_call_graph"],
    "analyze-data-flow": ["analyze-data-flow", "analyze_data_flow"],
    "checkout-status": ["checkout-status", "checkout_status"],
    "checkout-program": ["checkout-program", "checkout_program"],
    "checkin-program": ["checkin-program", "checkin_program"],
}

def find_free_port() -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return int(sock.getsockname()[1])

def normalize_mcp_url(url: str) -> str:
    normalized = url.strip().rstrip("/")
    if not normalized:
        return normalized
    if normalized.endswith("/mcp") or normalized.endswith("/mcp/message"):
        return normalized
    return normalized + "/mcp"

def resolve_live_tool_name(name: str) -> str:
    candidates = TOOL_ALIASES.get(name, [name, name.replace("-", "_"), name.replace("_", "-")])
    for candidate in candidates:
        if candidate in TOOL_NAMES:
            return candidate
    return name

def read_server_log_tail(lines: int = 60) -> str:
    log_path = SERVER_INFO.get("log_path")
    if not log_path:
        return ""
    path = Path(log_path)
    if not path.exists():
        return ""
    return "\n".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-lines:])

def payload_texts(payload: Any) -> list[str]:
    if not isinstance(payload, dict):
        return []
    content = payload.get("content")
    if not isinstance(content, list):
        return []
    texts = []
    for item in content:
        if isinstance(item, dict):
            text = item.get("text")
            if isinstance(text, str):
                texts.append(text)
    return texts

def detect_semantic_error(payload: Any) -> bool:
    texts = payload_texts(payload)
    joined = "\n".join(texts)
    if not joined:
        return False
    error_markers = [
        "## Error",
        '"success": false',
        "No program loaded",
        "Repository connection failed",
        "Authentication failed",
        "ReadOnlyException",
    ]
    return any(marker in joined for marker in error_markers)

def detect_shared_route_signal(payload: Any) -> bool:
    texts = payload_texts(payload)
    joined = "\n".join(texts)
    if not joined:
        return False
    route_markers = [
        "connect-shared-project",
        "Ghidra server not reachable",
        "serverHost",
        "serverPort",
        "127.0.0.1:13100",
    ]
    return any(marker in joined for marker in route_markers)

def start_local_server(port: int | None = None) -> dict[str, str]:
    global SERVER_PROCESS, SERVER_LOG_HANDLE, SERVER_INFO
    if SERVER_PROCESS is not None and SERVER_PROCESS.poll() is None:
        return SERVER_INFO
    chosen_port = find_free_port() if port is None else port
    runtime_project_dir = PROJECT_DIR / f"runtime_{chosen_port}"
    runtime_project_dir.mkdir(parents=True, exist_ok=True)
    log_path = SESSION_DIR / f"server_{chosen_port}.log"
    env = os.environ.copy()
    for key in [
        "AGENT_DECOMPILE_BACKEND_URL",
        "AGENT_DECOMPILE_MCP_SERVER_URL",
        "AGENT_DECOMPILE_GHIDRA_SERVER_USERNAME",
        "AGENT_DECOMPILE_GHIDRA_SERVER_PASSWORD",
        "AGENT_DECOMPILE_GHIDRA_SERVER_HOST",
        "AGENT_DECOMPILE_GHIDRA_SERVER_PORT",
        "AGENT_DECOMPILE_GHIDRA_SERVER_REPOSITORY",
        "AGENTDECOMPILE_SERVER_HOST",
        "AGENTDECOMPILE_SERVER_PORT",
        "AGENTDECOMPILE_SERVER_USERNAME",
        "AGENTDECOMPILE_SERVER_PASSWORD",
        "AGENTDECOMPILE_SERVER_REPOSITORY",
    ]:
        env.pop(key, None)
    cmd = [
        sys.executable,
        "-m",
        "agentdecompile_cli.server",
        "-t",
        "streamable-http",
        "--host",
        "127.0.0.1",
        "--port",
        str(chosen_port),
        "--project-path",
        str(runtime_project_dir),
        "--project-name",
        "usage_validation",
    ]
    SERVER_LOG_HANDLE = open(log_path, "w", encoding="utf-8")
    SERVER_PROCESS = subprocess.Popen(
        cmd,
        cwd=str(REPO_ROOT),
        stdout=SERVER_LOG_HANDLE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    SERVER_INFO = {
        "port": str(chosen_port),
        "url": f"http://127.0.0.1:{chosen_port}/mcp",
        "log_path": str(log_path),
        "project_path": str(runtime_project_dir),
        "command": " ".join(cmd),
    }
    return SERVER_INFO

async def wait_for_server(url: str, timeout: int = 120) -> bool:
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        if SERVER_PROCESS is not None and SERVER_PROCESS.poll() is not None:
            break
        try:
            async with AgentDecompileMcpClient(url=url) as client:
                await client.list_tools()
                return True
        except Exception as exc:
            last_error = exc
            await asyncio.sleep(1)
    if last_error is not None:
        print(f"Last connection error: {last_error}")
    return False

async def list_tools_live(url: str):
    async with AgentDecompileMcpClient(url=url) as client:
        return await client.list_tools()

async def list_resources_live(url: str):
    async with AgentDecompileMcpClient(url=url) as client:
        return await client.list_resources()

async def call_tool_live(url: str, name: str, arguments: dict[str, Any] | None = None):
    resolved_name = resolve_live_tool_name(name)
    async with AgentDecompileMcpClient(url=url) as client:
        return await client.call_tool(resolved_name, arguments or {})

async def record_tool(label: str, name: str, arguments: dict[str, Any] | None = None, expect_error: bool = False):
    started = time.time()
    error_text = None
    payload = None
    succeeded = False
    resolved_name = resolve_live_tool_name(name)
    semantic_error = False
    shared_route_signal = False
    try:
        payload = await call_tool_live(SERVER_INFO["url"], resolved_name, arguments or {})
        succeeded = True
        semantic_error = detect_semantic_error(payload)
        shared_route_signal = detect_shared_route_signal(payload)
    except Exception as exc:
        error_text = str(exc)
        payload = {"error": error_text}
    matched_expectation = ((not succeeded) or semantic_error) if expect_error else (succeeded and not semantic_error)
    entry = {
        "label": label,
        "tool": resolved_name,
        "arguments": arguments or {},
        "succeeded": succeeded,
        "expect_error": expect_error,
        "semantic_error": semantic_error,
        "shared_route_signal": shared_route_signal,
        "matched_expectation": matched_expectation,
        "duration_seconds": round(time.time() - started, 3),
        "payload": payload,
        "error": error_text,
    }
    RESULTS.append(entry)
    return entry

def extract_program_path(payload: Any) -> str | None:
    if not isinstance(payload, dict):
        return None
    for key in ["path", "programPath", "program_path", "checkedOutProgram"]:
        value = payload.get(key)
        if isinstance(value, str) and value.strip():
            return value
    return None

def first_function_identifier(payload: Any) -> str | None:
    if not isinstance(payload, dict):
        return None
    functions = payload.get("functions")
    if isinstance(functions, list) and functions:
        first = functions[0]
        if isinstance(first, dict):
            for key in ["address", "name"]:
                value = first.get(key)
                if isinstance(value, str) and value.strip():
                    return value
    return None

def stop_local_server() -> None:
    global SERVER_PROCESS, SERVER_LOG_HANDLE
    if SERVER_PROCESS is not None and SERVER_PROCESS.poll() is None:
        SERVER_PROCESS.terminate()
        try:
            SERVER_PROCESS.wait(timeout=10)
        except subprocess.TimeoutExpired:
            SERVER_PROCESS.kill()
            SERVER_PROCESS.wait(timeout=10)
    SERVER_PROCESS = None
    if SERVER_LOG_HANDLE is not None:
        SERVER_LOG_HANDLE.close()
        SERVER_LOG_HANDLE = None

In [5]:
RESULTS.clear()
CURRENT_PROGRAM_PATH = None

# Allow reusing an existing server (e.g. set AGENTDECOMPILE_TEST_SERVER_URL=http://127.0.0.1:8765)
# Or set it directly here for notebook convenience:
if "AGENTDECOMPILE_TEST_SERVER_URL" not in os.environ:
    os.environ["AGENTDECOMPILE_TEST_SERVER_URL"] = "http://127.0.0.1:8765"  # change or clear as needed

EXTERNAL_SERVER_URL = os.environ.get("AGENTDECOMPILE_TEST_SERVER_URL", "").strip()
if EXTERNAL_SERVER_URL:
    normalized_external_url = normalize_mcp_url(EXTERNAL_SERVER_URL)
    SERVER_INFO = {
        "port": EXTERNAL_SERVER_URL.rsplit(":", 1)[-1].split("/")[0],
        "url": normalized_external_url,
        "log_path": "",
        "project_path": "",
        "command": f"(external) {EXTERNAL_SERVER_URL}",
    }
    show_note(
        "Using external server",
        f"URL: <code>{html.escape(SERVER_INFO['url'])}</code><br>Compatibility path: <code>{html.escape(normalized_external_url + '/message')}</code>",
        color="#1f6f43",
    )
    SERVER_READY = await wait_for_server(SERVER_INFO["url"], timeout=30)
else:
    SERVER_INFO = start_local_server()
    show_note(
        "Starting local server",
        f"Command: <code>{html.escape(SERVER_INFO['command'])}</code><br>URL: <code>{html.escape(SERVER_INFO['url'])}</code><br>Compatibility path: <code>{html.escape(SERVER_INFO['url'] + '/message')}</code>",
        color="#6a3fb5",
    )
    SERVER_READY = await wait_for_server(SERVER_INFO["url"], timeout=120)

if not SERVER_READY:
    raise RuntimeError(
        "Local server did not become ready. Check the server log tail below.\n\n" + read_server_log_tail()
    )

live_tools = await list_tools_live(SERVER_INFO["url"])
live_resources = await list_resources_live(SERVER_INFO["url"])

def tool_name(tool: Any) -> str:
    if isinstance(tool, dict):
        return str(tool.get("name", ""))
    if hasattr(tool, "name"):
        return str(tool.name)
    return str(tool)

TOOL_NAMES = sorted(name for name in (tool_name(tool) for tool in live_tools) if name)
OPEN_TOOL = resolve_live_tool_name("open-project")

display(Markdown("## Live Tool Discovery"))
as_json({
    "tool_count": len(TOOL_NAMES),
    "resource_count": len(live_resources),
    "open_tool": OPEN_TOOL,
    "tool_sample": TOOL_NAMES[:20],
    "server_url": SERVER_INFO["url"],
})

2026-03-11 11:55:25,486 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:55:25,489 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-11 11:55:25,496 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:55:25,972 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:55:25,975 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-11 11:55:25,981 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:55:26,439 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:55:26,441 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03

## Live Tool Discovery

<IPython.core.display.JSON object>

In [6]:
function_related_tools = [name for name in TOOL_NAMES if "function" in name]
as_json({
    "function_related_tools": function_related_tools,
    "has_list_functions": any(name in TOOL_NAMES for name in ["list-functions", "list_functions"]),
    "has_get_functions": any(name in TOOL_NAMES for name in ["get-functions", "get_functions"]),
    "has_decompile_function": any(name in TOOL_NAMES for name in ["decompile-function", "decompile_function"]),
})

<IPython.core.display.JSON object>

In [7]:
if OPEN_TOOL is None:
    raise RuntimeError("No open tool was advertised by the local server.")

open_entry = await record_tool(
    "Open local binary",
    OPEN_TOOL,
    {"path": str(PRIMARY_BINARY)},
)
project_listing_entry = await record_tool(
    "List project files",
    "list-project-files",
    {},
)
current_program_entry = await record_tool(
    "Get current program",
    "get-current-program",
    {},
)

CURRENT_PROGRAM_PATH = extract_program_path(current_program_entry["payload"])
display(Markdown("## Local Open Flow"))
as_json(
    {
        "open_succeeded": open_entry["succeeded"],
        "current_program_path": CURRENT_PROGRAM_PATH,
        "project_listing_succeeded": project_listing_entry["succeeded"],
        "open_payload": open_entry["payload"],
        "current_program_payload": current_program_entry["payload"],
    }
)

2026-03-11 11:56:11,293 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:56:11,296 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-11 11:56:11,534 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:56:11,987 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:56:11,989 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-11 11:56:11,993 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:56:12,454 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:56:12,457 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03

## Local Open Flow

<IPython.core.display.JSON object>

In [8]:
def with_program(arguments: dict[str, Any] | None = None) -> dict[str, Any]:
    merged = dict(arguments or {})
    if CURRENT_PROGRAM_PATH and "programPath" not in merged and "program_path" not in merged:
        merged["programPath"] = CURRENT_PROGRAM_PATH
    return merged

functions_entry = await record_tool(
    "List functions sample",
    "list-functions",
    with_program({"limit": 5}),
)
search_entry = await record_tool(
    "Search symbols by name",
    "search-symbols-by-name",
    with_program({"query": "main", "max_results": 5}),
)

sample_identifier = first_function_identifier(functions_entry["payload"])
references_target = sample_identifier or "entry"
references_to_entry = await record_tool(
    "References to sample target",
    "get-references",
    with_program({"mode": "to", "target": references_target, "limit": 10}),
)

display(Markdown("## Read-Only Inspection Flows"))
as_json(
    {
        "function_identifier_used": sample_identifier,
        "functions_payload": functions_entry["payload"],
        "search_payload": search_entry["payload"],
        "references_payload": references_to_entry["payload"],
    }
)

2026-03-11 11:56:31,045 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:56:31,051 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-11 11:56:31,059 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:56:31,583 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:56:31,586 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-11 11:56:31,595 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:56:32,073 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:56:32,077 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03

## Read-Only Inspection Flows

<IPython.core.display.JSON object>

In [9]:
payload_preview = str(functions_entry["payload"])
if len(payload_preview) > 500:
    payload_preview = payload_preview[:497] + "..."
as_json({
    "functions_entry_tool": functions_entry["tool"],
    "functions_entry_succeeded": functions_entry["succeeded"],
    "functions_entry_payload_preview": payload_preview,
})

<IPython.core.display.JSON object>

In [10]:
export_entry = await record_tool(
    "Export HTML",
    "export",
    with_program({
        "format": "html",
        "outputPath": str(OUTPUT_DIR / "program.html"),
    }),
)
imports_entry = await record_tool(
    "List imports",
    "list-imports",
    with_program({"limit": 5}),
)
exports_entry = await record_tool(
    "List exports",
    "list-exports",
    with_program({"limit": 5}),
)
strings_entry = await record_tool(
    "Manage strings regex",
    "manage-strings",
    with_program({
        "mode": "regex",
        "query": "Agent|Test|ELF",
        "include_referencing_functions": True,
        "limit": 20,
    }),
)
constants_entry = await record_tool(
    "Search constants",
    "search-constants",
    with_program({"mode": "specific", "value": 32, "max_results": 20}),
)

advanced_summary = {
    "export_payload": export_entry["payload"],
    "imports_payload": imports_entry["payload"],
    "exports_payload": exports_entry["payload"],
    "strings_payload": strings_entry["payload"],
    "constants_payload": constants_entry["payload"],
    "export_exists": (OUTPUT_DIR / "program.html").exists(),
}

if sample_identifier:
    info_entry = await record_tool(
        "Get function info",
        "get-functions",
        with_program({
            "identifier": sample_identifier,
            "view": "info",
            "include_callers": True,
            "include_callees": True,
        }),
    )
    decompile_entry = await record_tool(
        "Get function decompile",
        "get-functions",
        with_program({"identifier": sample_identifier, "view": "decompile"}),
    )
    disassemble_entry = await record_tool(
        "Get function disassemble",
        "get-functions",
        with_program({"identifier": sample_identifier, "view": "disassemble"}),
    )
    call_graph_entry = await record_tool(
        "Get call graph",
        "get-call-graph",
        with_program({
            "function_identifier": sample_identifier,
            "mode": "callees",
            "max_depth": 2,
        }),
    )
    references_from_entry = await record_tool(
        "References from sample function",
        "get-references",
        with_program({"mode": "from", "target": sample_identifier, "limit": 25}),
    )
    data_flow_entry = await record_tool(
        "Analyze data flow",
        "analyze-data-flow",
        with_program({
            "function_address": sample_identifier,
            "start_address": sample_identifier,
            "direction": "forward",
        }),
    )
    advanced_summary["function_detail_payloads"] = {
        "info": info_entry["payload"],
        "decompile": decompile_entry["payload"],
        "disassemble": disassemble_entry["payload"],
        "call_graph": call_graph_entry["payload"],
        "references_from": references_from_entry["payload"],
        "data_flow": data_flow_entry["payload"],
    }
else:
    advanced_summary["function_detail_payloads"] = "Skipped: no sample function identifier was available."

display(Markdown("## Extended Local Tool Flows"))
as_json(advanced_summary)

2026-03-11 11:57:06,465 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:57:06,468 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-11 11:57:06,496 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:57:07,063 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:57:07,066 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-11 11:57:07,072 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:57:07,587 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:57:07,600 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03

## Extended Local Tool Flows

<IPython.core.display.JSON object>

In [13]:
version_control_probe_summary = "Skipped by widget flag."
if RUN_VERSION_CONTROL_PROBES.value:
    checkout_status_entry = await record_tool(
        "Checkout status in local mode",
        "checkout-status",
        with_program({}),
        expect_error=True,
    )
    checkout_entry = await record_tool(
        "Checkout program in local mode",
        "checkout-program",
        with_program({"exclusive": False}),
        expect_error=True,
    )
    checkin_entry = await record_tool(
        "Checkin program in local mode",
        "checkin-program",
        with_program({
            "comment": "Notebook local-mode behavior probe",
            "keep_checked_out": False,
        }),
        expect_error=True,
    )
    version_control_probe_summary = {
        "checkout_status": checkout_status_entry["payload"],
        "checkout": checkout_entry["payload"],
        "checkin": checkin_entry["payload"],
    }

display(Markdown("## Explicit Version-Control Behavior Probes"))
as_json(version_control_probe_summary)

2026-03-11 11:57:48,752 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:57:48,755 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-11 11:57:48,760 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:57:49,239 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:57:49,241 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03-11 11:57:49,245 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:57:49,728 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 200 OK"
2026-03-11 11:57:49,732 [INFO] HTTP Request: POST http://127.0.0.1:8765/mcp/message "HTTP/1.1 202 Accepted"
Backend: AgentDecompile v1.1.0 (protocol 2025-03-26)
2026-03

## Explicit Version-Control Behavior Probes

<IPython.core.display.JSON object>

## Shared-Server Workflows

 > These examples stay visible for onboarding, but they are intentionally not executed by default in this notebook.

Shared-server commands in [USAGE.md] assume external credentials and a live Ghidra shared repository. This notebook keeps them as documentation-only examples so the default experience remains local, reproducible, and read-only.

### Intended semantic rule captured by this notebook

- Local projects should remain read-only by default.
- `checkout-program` and `checkin-program` are explicit operations, not implicit side effects of opening a local binary.
- If version control is unsupported in the current context, the failure should be early and clear rather than silently promoting the project into a writable shared-server state.

## Unique Pattern Mining From Terminal Artifacts

This section extracts recurring behavior signatures from captured terminal logs and turns them into explicit validation artifacts.

Patterns mined here:

- endpoint usage and normalization (`/mcp` vs base URL forms)
- stateless session symptoms (`No program loaded`)
- shared-server authentication fingerprints (`NotConnectedException` + `FailedLoginException`)
- command-surface mismatch hints (for example unknown CLI options)

The output is both human-readable and machine-consumable so docs and tests can reuse the same facts.

In [ ]:
TERMINAL_LOG_PATH = REPO_ROOT / "terminal_output.txt"

pattern_report = {
    "log_path": str(TERMINAL_LOG_PATH),
    "exists": TERMINAL_LOG_PATH.exists(),
    "line_count": 0,
    "endpoint_hits": {
        "canonical_mcp": 0,
        "compat_mcp_message": 0,
        "base_url_only": 0,
    },
    "session_state_signals": {
        "no_program_loaded": 0,
        "current_program_unknown": 0,
    },
    "auth_failure_signals": {
        "failed_login_exception": 0,
        "not_connected_exception": 0,
        "authentication_failed": 0,
    },
    "cli_surface_signals": {
        "unknown_option": 0,
    },
}

if TERMINAL_LOG_PATH.exists():
    log_text = TERMINAL_LOG_PATH.read_text(encoding="utf-8", errors="replace")
    lines = log_text.splitlines()
    pattern_report["line_count"] = len(lines)

    # Endpoint usage profile
    pattern_report["endpoint_hits"]["canonical_mcp"] = log_text.count("/mcp")
    pattern_report["endpoint_hits"]["compat_mcp_message"] = log_text.count("/mcp/message")
    pattern_report["endpoint_hits"]["base_url_only"] = len(re.findall(r"--server-url\s+http://[^\s'\"]+/?(?!mcp)", log_text))

    # Session-state symptoms
    pattern_report["session_state_signals"]["no_program_loaded"] = log_text.count("No program loaded")
    pattern_report["session_state_signals"]["current_program_unknown"] = log_text.count("**Name:** `unknown`")

    # Shared auth fingerprints
    pattern_report["auth_failure_signals"]["failed_login_exception"] = log_text.count("FailedLoginException")
    pattern_report["auth_failure_signals"]["not_connected_exception"] = log_text.count("NotConnectedException")
    pattern_report["auth_failure_signals"]["authentication_failed"] = log_text.count("Authentication failed")

    # Command-surface mismatches
    pattern_report["cli_surface_signals"]["unknown_option"] = log_text.count("No such option")

PATTERN_REPORT = pattern_report

show_note(
    "Pattern mining complete",
    "Extracted endpoint, session, auth, and CLI-surface signatures from terminal_output.txt.",
    color="#1f6f43",
)
as_json(PATTERN_REPORT)

In [ ]:
tmp_compact_files = sorted((REPO_ROOT / "tmp" / "usage_validation").glob("*/compact_summary.json"))

def parse_compact_file(path: Path) -> dict[str, Any]:
    data = json.loads(path.read_text(encoding="utf-8"))
    rows = data.get("version_control_rows", [])
    semantic_error_rows = 0
    succeeded_with_error_text = 0
    for row in rows:
        payload = row.get("payload", {})
        text_blob = "\n".join(payload_texts(payload))
        if detect_semantic_error(payload):
            semantic_error_rows += 1
        if row.get("succeeded") and ("## Error" in text_blob or "ReadOnlyException" in text_blob):
            succeeded_with_error_text += 1
    return {
        "file": str(path),
        "version_control_rows": len(rows),
        "semantic_error_rows": semantic_error_rows,
        "succeeded_with_error_text": succeeded_with_error_text,
    }

tmp_compact_report = [parse_compact_file(path) for path in tmp_compact_files]

pattern_invariants = {
    "tmp_compact_files_found": len(tmp_compact_files),
    "tmp_compact_report": tmp_compact_report,
    "session_semantic_error_rows": sum(1 for item in RESULTS if item.get("semantic_error")),
    "session_shared_route_signals": sum(1 for item in RESULTS if item.get("shared_route_signal")),
    "session_expectation_mismatches": sum(1 for item in RESULTS if not item.get("matched_expectation", False)),
}

show_note(
    "TMP invariant extraction",
    "Parsed compact summaries under tmp/usage_validation and merged with current session semantic-error signals.",
    color="#2855aa",
)
as_json(pattern_invariants)

In [14]:
def summarize_results() -> list[dict[str, Any]]:
    rows = []
    for entry in RESULTS:
        payload_text = json.dumps(entry["payload"], default=str)
        if len(payload_text) > 220 and not current_flags()["verbose_tool_output"]:
            payload_text = payload_text[:217] + "..."
        rows.append(
            {
                "label": entry["label"],
                "tool": entry["tool"],
                "succeeded": entry["succeeded"],
                "semantic_error": entry.get("semantic_error", False),
                "shared_route_signal": entry.get("shared_route_signal", False),
                "expect_error": entry["expect_error"],
                "matched_expectation": entry["matched_expectation"],
                "duration_seconds": entry["duration_seconds"],
                "payload_preview": payload_text,
            }
        )
    return rows

summary_rows = summarize_results()
summary_table_rows = "".join(
    "<tr>"
    f"<td style='padding:0.35rem 0.5rem'>{html.escape(str(row['label']))}</td>"
    f"<td style='padding:0.35rem 0.5rem'><code>{html.escape(str(row['tool']))}</code></td>"
    f"<td style='padding:0.35rem 0.5rem'>{row['succeeded']}</td>"
    f"<td style='padding:0.35rem 0.5rem'>{row['semantic_error']}</td>"
    f"<td style='padding:0.35rem 0.5rem'>{row['shared_route_signal']}</td>"
    f"<td style='padding:0.35rem 0.5rem'>{row['expect_error']}</td>"
    f"<td style='padding:0.35rem 0.5rem'>{row['matched_expectation']}</td>"
    f"<td style='padding:0.35rem 0.5rem'>{row['duration_seconds']}</td>"
    f"<td style='padding:0.35rem 0.5rem'><code>{html.escape(str(row['payload_preview']))}</code></td>"
    "</tr>"
    for row in summary_rows
)

display(Markdown("## Execution Summary"))
display(
    HTML(
        "<table style='border-collapse:collapse; width:100%'>"
        "<thead><tr>"
        "<th style='text-align:left; padding:0.35rem 0.5rem'>Label</th>"
        "<th style='text-align:left; padding:0.35rem 0.5rem'>Tool</th>"
        "<th style='text-align:left; padding:0.35rem 0.5rem'>Succeeded</th>"
        "<th style='text-align:left; padding:0.35rem 0.5rem'>Semantic error</th>"
        "<th style='text-align:left; padding:0.35rem 0.5rem'>Shared-route signal</th>"
        "<th style='text-align:left; padding:0.35rem 0.5rem'>Expect error</th>"
        "<th style='text-align:left; padding:0.35rem 0.5rem'>Matched expectation</th>"
        "<th style='text-align:left; padding:0.35rem 0.5rem'>Seconds</th>"
        "<th style='text-align:left; padding:0.35rem 0.5rem'>Payload preview</th>"
        "</tr></thead>"
        f"<tbody>{summary_table_rows}</tbody></table>"
    )
)

show_note(
    "Interpretation",
    "Rows where <code>semantic_error</code> is true indicate domain-level failures wrapped in successful transport/tool envelopes. Rows with <code>shared_route_signal</code> indicate local flows that surfaced shared-server resolution hints.",
    color="#8a1c1c",
)

## Execution Summary

Label,Tool,Succeeded,Expect error,Matched expectation,Seconds,Payload preview
Open local binary,open_project,True,False,True,0.764,"{""content"": [{""type"": ""text"", ""text"": ""## Project (import)\n\n**operation:** import\n**importedFrom:** C:\\GitHub\\agentdecompile\\tests\\fixtures\\test_x86_64\n**filesDiscovered:** 1\n**filesImported:** 1\n\n### Impo..."
List project files,list_project_files,True,False,True,0.459,"{""content"": [{""type"": ""text"", ""text"": ""## Project Files\n\n**Folder:** `/`\n**Count:** 0\n\n\n### About This Tool\n\nLists files in the current Ghidra project directory.""}], ""isError"": false}"
Get current program,get_current_program,True,False,True,0.468,"{""content"": [{""type"": ""text"", ""text"": ""## Current Program\n\n**Name:** `test_x86_64`\n**Path:** ``\n**Language:** x86:LE:64:default\n**Compiler:** gcc\n**Image Base:** ``\n**Functions:** 3\n**Symbols:** 0\n\n### About..."
List functions sample,list_functions,True,False,True,0.594,"{""content"": [{""type"": ""text"", ""text"": ""## Function Listing\n\nShowing **2** of **2** results (offset 0).\n\n| Name | Address | Size | Params | External | Thunk |\n| --- | --- | --- | --- | --- | --- |\n| entry | 10000..."
Search symbols by name,search-symbols-by-name,True,False,True,0.535,"{""content"": [{""type"": ""text"", ""text"": ""## Symbol Listing\n\nShowing **2** of **2** results (offset 0).\n\n| Name | Address | Type | Namespace |\n| --- | --- | --- | --- |\n| _main | 1000004b0 | Label | Global |\n| s__..."
References to sample target,get_references,True,False,True,0.486,"{""content"": [{""type"": ""text"", ""text"": ""## Getreferences (to)\n\n**mode:** to\n**target:** 1000004b0\n\n### References\n\n| fromAddress | toAddress | type | function |\n| --- | --- | --- | --- |\n| Entry Point | 100000..."
Export HTML,export,True,False,True,0.738,"{""content"": [{""type"": ""text"", ""text"": ""## Export Results\n\n**Format:** html\n**Output Path:** `C:\\GitHub\\agentdecompile\\tmp\\usage_validation\\usage_validation_yyg9uxnk\\exports\\program.html`\n**Status:** Success..."
List imports,list_imports,True,False,True,0.576,"{""content"": [{""type"": ""text"", ""text"": ""## Import Listing\n\nShowing **1** of **1** results (offset 0).\n\n| Name | Address | Namespace |\n| --- | --- | --- |\n| _printf | EXTERNAL:00000001 | /usr/lib/libSystem.B.dylib..."
List exports,list_exports,True,False,True,0.533,"{""content"": [{""type"": ""text"", ""text"": ""## Export Listing\n\nShowing **5** of **6** results (offset 0). More results available \u2014 use `offset=5` to continue.\n\n| Name | Address |\n| --- | --- |\n| __mh_execute_hea..."
Manage strings regex,manage-strings,True,False,True,0.537,"{""content"": [{""type"": ""text"", ""text"": ""## Strings\n\nShowing **1** of **1** results (offset 0).\n\n| Address | Value | References |\n| --- | --- | --- |\n| 100000520 | ReVa Test Program\n | 0 |\n\n### About This Tool\..."


In [15]:
def compact_payload(payload: Any) -> Any:
    if isinstance(payload, dict):
        return {key: compact_payload(value) for key, value in list(payload.items())[:10]}
    if isinstance(payload, list):
        return [compact_payload(item) for item in payload[:5]]
    return payload

compact_summary = {
    "open_entry": {
        "tool": open_entry["tool"],
        "succeeded": open_entry["succeeded"],
        "payload": compact_payload(open_entry["payload"]),
    },
    "current_program_entry": {
        "tool": current_program_entry["tool"],
        "succeeded": current_program_entry["succeeded"],
        "payload": compact_payload(current_program_entry["payload"]),
    },
    "functions_entry": {
        "tool": functions_entry["tool"],
        "succeeded": functions_entry["succeeded"],
        "payload": compact_payload(functions_entry["payload"]),
    },
    "export_entry": {
        "tool": export_entry["tool"],
        "succeeded": export_entry["succeeded"],
        "payload": compact_payload(export_entry["payload"]),
    },
    "version_control_rows": [
        {
            "label": entry["label"],
            "tool": entry["tool"],
            "succeeded": entry["succeeded"],
            "expect_error": entry["expect_error"],
            "matched_expectation": entry["matched_expectation"],
            "payload": compact_payload(entry["payload"]),
        }
        for entry in RESULTS
        if "checkout" in entry["tool"] or "checkin" in entry["tool"]
    ],
}
as_json(compact_summary)

<IPython.core.display.JSON object>

In [16]:
summary_artifact = SESSION_DIR / "compact_summary.json"
summary_artifact.write_text(json.dumps(compact_summary, indent=2, default=str), encoding="utf-8")
print(summary_artifact)

C:\GitHub\agentdecompile\tmp\usage_validation\usage_validation_yyg9uxnk\compact_summary.json


In [ ]:
if not EXTERNAL_SERVER_URL:
    stop_local_server()
    show_note(
        "Cleanup complete",
        f"Temporary workspace retained for inspection at <code>{html.escape(str(SESSION_DIR))}</code>.",
        color="#1f6f43",
    )
    print(read_server_log_tail(40))
else:
    show_note(
        "Cleanup complete (external server retained)",
        f"External server at <code>{html.escape(EXTERNAL_SERVER_URL)}</code> was not stopped.",
        color="#1f6f43",
    )

: 